# AgroSense — Entrenamiento del modelo de ganado (Colab)

Prueba de concepto del módulo **ganadero** (tareas 28–29 del cronograma): clasificar fotos de
ganado bovino en **sano** o con **dermatosis nodular contagiosa** (*Lumpy Skin Disease*, LSD).
Sigue el mismo esquema que el notebook de cultivos: transfer learning con MobileNetV2 y
exportación a `.tflite` para la app Kotlin.

**Dataset:** *Lumpy Skin Images Dataset* — Kumar, S. & Shastri, S. (2022), Universidad de Jammu.
Mendeley Data, v1. https://doi.org/10.17632/w36hpf86j2.1 — licencia **CC BY 4.0** (se puede usar,
incluso con fines comerciales, citando a los autores). Trae 324 imágenes con LSD y 700 normales,
PNG de 256×256. Se baja de su espejo en Kaggle (`warcoder/lumpy-skin-images-dataset`).

### En qué se diferencia del notebook de cultivos, y por qué

| | Cultivos | Ganado |
|---|---|---|
| Tamaño | ~24 000 imágenes | ~1 000 imágenes |
| Balance | más o menos parejo | el doble de sanas que de enfermas |
| Limpieza | — | duplicados exactos, **casi-duplicados** y **revisión visual** |
| Split | `validation_split` + mitad de lotes | **estratificado por archivo**, 70/15/15 |
| Desbalance | — | **pesos por clase** en el entrenamiento |
| Métrica principal | accuracy | **sensibilidad para LSD** y accuracy balanceada |
| Augmentation | ligero | más fuerte, porque hay pocas fotos |
| Salida | `agrosense_crop_model.tflite` + `labels.txt` | **`agrosense_ganado_model.tflite` + `labels_ganado.txt`** |

**No se usa `tflite-support` ni metadata.** La app carga los modelos con el `Interpreter` de
TensorFlow Lite, normaliza los píxeles a `[-1, 1]` ella misma y lee las etiquetas de un `.txt`.
Este notebook respeta ese contrato y lo verifica en la sección 7.

Flujo:
1. Setup
2. Descarga del dataset (+ fotos extra opcionales)
3. Limpieza (duplicados exactos, casi-duplicados, revisión visual) y split estratificado
4. Pipeline de datos
5. Modelo: MobileNetV2 — fase *freeze* → fase *fine-tune*
6. Evaluación
7. Exportar `.tflite` y verificarlo como lo usará la app
8. Siguiente paso en Android Studio

## 1. Setup

In [ ]:
# Librerías. TensorFlow, numpy, scikit-learn y matplotlib ya vienen en Colab.
import glob
import hashlib
import json
import os
import random
import re
import shutil

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU disponible:", tf.config.list_physical_devices('GPU'))

In [ ]:
# Drive solo guarda lo que debe sobrevivir entre sesiones: checkpoints, el manifiesto del split
# y los archivos exportados. Las imágenes se trabajan en el disco local de la VM: leer miles de
# archivos chicos desde Drive lo desconecta ("Transport endpoint is not connected"), como pasó
# con el notebook de cultivos. Este dataset pesa ~160 MB, así que se vuelve a bajar en cada sesión.
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/AgroSense'
GANADO_DIR = os.path.join(PROJECT_DIR, 'ganado')
CKPT_DIR = os.path.join(GANADO_DIR, 'checkpoints')
EXPORT_DIR = os.path.join(PROJECT_DIR, 'export')   # la misma que cultivos; los nombres no chocan

LOCAL_DIR = '/content/ganado'                       # disco local de la VM
SPLIT_DIR = os.path.join(LOCAL_DIR, 'split')

for d in [GANADO_DIR, CKPT_DIR, EXPORT_DIR, LOCAL_DIR]:
    os.makedirs(d, exist_ok=True)

print("Proyecto en Drive:", GANADO_DIR)

## 2. Descargar el dataset

Igual que en cultivos: `kagglehub` con tu `kaggle.json` (Kaggle → Settings → API → Create New
Token). Si Colab ya te lo pidió en el notebook de cultivos, en una sesión nueva te lo vuelve a pedir.

**Si el espejo de Kaggle deja de existir**, baja el zip directo de Mendeley
(https://data.mendeley.com/datasets/w36hpf86j2/1 → *Download All*), descomprímelo en Colab y pon
esa carpeta en `RAW_DIR` en lugar de la línea de `kagglehub`. El resto del notebook no cambia.

In [ ]:
!pip install -q kagglehub

import kagglehub

RAW_DIR = kagglehub.dataset_download("warcoder/lumpy-skin-images-dataset")
print("Descargado en:", RAW_DIR)

EXTENSIONES = {'.jpg', '.jpeg', '.png'}

def listar_imagenes(carpeta):
    return sorted(
        os.path.join(carpeta, f) for f in os.listdir(carpeta)
        if os.path.splitext(f)[1].lower() in EXTENSIONES
    )

# Toda carpeta que contenga imágenes cuenta como una clase. Si el zip trae la misma clase en
# dos rutas (pasa con algunas subidas a Kaggle), se acumulan y la sección 3 quita los duplicados.
carpetas = {}
for raiz, _dirs, _archivos in os.walk(RAW_DIR):
    imagenes = listar_imagenes(raiz)
    if imagenes:
        carpetas.setdefault(os.path.basename(raiz), []).extend(imagenes)

print(f"\n{len(carpetas)} carpetas con imágenes:")
for nombre, imagenes in sorted(carpetas.items()):
    print(f"  {len(imagenes):5d}  {nombre}")

### 2.1 Fotos extra (opcional)

El Grad-CAM de la sección 6.1 mostró que el modelo usa el **encuadre y el entorno** como atajo:
vaca lejos o en pasto → "sana"; corral, personas o postes → "LSD". Este dataset casi no tiene vacas
sanas fotografiadas de cerca o en corral, que es justo lo que rompe ese atajo.

Si consiguen fotos así (propias, o de un dataset con licencia que lo permita), súbanlas a Drive en
`AgroSense/ganado/extra/<carpeta>/`, con el **mismo nombre de carpeta del dataset**:
`Normal Skin` para sanas y `Lumpy Skin` para enfermas. Esta celda las suma; si la carpeta no
existe, no hace nada.

In [ ]:
EXTRA_DIR = os.path.join(GANADO_DIR, 'extra')

def clave(ruta):
    # Identificador estable de cada imagen, igual en todas las sesiones. Se usa en el manifiesto
    # del split y en la lista de exclusiones.
    for prefijo, base in (('dataset', RAW_DIR), ('extra', EXTRA_DIR)):
        rel = os.path.relpath(ruta, base)
        if not rel.startswith('..'):
            return f"{prefijo}/{rel}".replace(os.sep, '/')
    return ruta

if os.path.isdir(EXTRA_DIR):
    for nombre in sorted(os.listdir(EXTRA_DIR)):
        carpeta = os.path.join(EXTRA_DIR, nombre)
        if os.path.isdir(carpeta):
            extra = listar_imagenes(carpeta)
            carpetas.setdefault(nombre, []).extend(extra)
            print(f"  +{len(extra):4d}  fotos extra en '{nombre}'")
else:
    print("Sin fotos extra (no existe", EXTRA_DIR + ").")

## 3. Limpieza y split estratificado

### Etiquetas

Las carpetas del dataset se renombran al estilo de PlantVillage (`Especie___clase`), que es como la
app ya muestra los diagnósticos de cultivos. Este nombre es lo que se guarda en Room y lo que verá
el usuario.

> ⚠️ El **orden** de las etiquetas lo fija `image_dataset_from_directory` (alfabético) y la app lo
> lee por índice de `labels_ganado.txt`. Ese archivo lo genera la sección 7; **nunca lo escribas a
> mano.**

### Duplicados

Con tan pocas imágenes, una foto repetida que cae una vez en entrenamiento y otra en prueba infla
el accuracy sin que se note. Se comparan por contenido (MD5), no por nombre: se deja una copia de
cada imagen, y si la misma imagen aparece con las dos etiquetas, se descarta.

### Split

70 % entrenamiento, 15 % validación (para *early stopping*) y 15 % prueba (no se toca hasta la
sección 6). Se reparte **por clase**, para que los tres conjuntos tengan la misma proporción de
sanas y enfermas. La lista de archivos de cada conjunto se guarda en Drive, así que el número
de la presentación se puede reproducir.

In [ ]:
# Nombre de carpeta del dataset (normalizado) -> etiqueta del modelo.
MAPEO_CLASES = {
    'lumpy skin':  'Cattle___Lumpy_skin_disease',
    'normal skin': 'Cattle___healthy',
}

def _norm(s):
    # Tolera mayúsculas, guiones bajos y espacios de más ("Normal_Skin", "normal skin ").
    return re.sub(r'[\s_\-]+', ' ', s.strip().lower())

MAPEO_NORM = {_norm(k): v for k, v in MAPEO_CLASES.items()}

no_reconocidas = sorted(c for c in carpetas if _norm(c) not in MAPEO_NORM)
if no_reconocidas:
    print("Carpetas ignoradas (no están en MAPEO_CLASES):", no_reconocidas)

encontradas = {MAPEO_NORM[_norm(c)] for c in carpetas if _norm(c) in MAPEO_NORM}
faltan = set(MAPEO_CLASES.values()) - encontradas
assert not faltan, (
    f"No se encontraron carpetas para: {sorted(faltan)}.\n"
    "Revisa la lista impresa arriba y ajusta las llaves de MAPEO_CLASES."
)

def md5(ruta):
    h = hashlib.md5()
    with open(ruta, 'rb') as f:
        for bloque in iter(lambda: f.read(1 << 20), b''):
            h.update(bloque)
    return h.hexdigest()

etiquetas_por_hash = {}
ruta_por_hash = {}
total_leidas = 0
for carpeta, imagenes in carpetas.items():
    etiqueta = MAPEO_NORM.get(_norm(carpeta))
    if etiqueta is None:
        continue
    for ruta in imagenes:
        total_leidas += 1
        h = md5(ruta)
        etiquetas_por_hash.setdefault(h, set()).add(etiqueta)
        ruta_por_hash.setdefault(h, ruta)

conflictivas = [h for h, e in etiquetas_por_hash.items() if len(e) > 1]
por_clase = {}
for h, etiquetas in etiquetas_por_hash.items():
    if len(etiquetas) == 1:
        por_clase.setdefault(next(iter(etiquetas)), []).append(ruta_por_hash[h])

unicas = sum(len(v) for v in por_clase.values())
print(f"Imágenes leídas:            {total_leidas}")
print(f"Duplicadas eliminadas:      {total_leidas - len(etiquetas_por_hash)}")
print(f"Con las dos etiquetas (se descartan): {len(conflictivas)}")
print(f"Imágenes únicas para usar:  {unicas}\n")
for etiqueta, rutas in sorted(por_clase.items()):
    print(f"  {len(rutas):5d}  {etiqueta}  ({len(rutas) / unicas:.0%})")

### 3.1 Casi-duplicados

El MD5 solo detecta archivos **idénticos byte a byte**. La misma foto recortada, redimensionada
o vuelta a comprimir tiene otro MD5, y el Grad-CAM mostró un caso así: la escena del hombre de
camisa azul junto a la vaca aparecía una vez como sana y otra como enferma.

Se hace en dos pasos, porque ninguno sirve solo:

1. **Candidatos por parecido.** Cada foto pasa por MobileNetV2 (pesos de ImageNet, no los del
   modelo entrenado) y se buscan sus 10 vecinas más parecidas. Esto encuentra la misma foto
   aunque esté recortada, pero también fotos distintas de escenas parecidas. Por eso no basta.
2. **Verificación geométrica.** Para cada candidata se buscan puntos característicos (ORB) en las
   dos imágenes y se comprueba si encajan con un solo recorte, cambio de escala o rotación
   (RANSAC). La misma foto da decenas o cientos de puntos que encajan; dos fotos distintas, casi
   ninguno. Se consideran la misma si encajan al menos `MIN_COINCIDENCIAS`.

Cuando dos fotos son la misma:
- **misma etiqueta:** se deja una;
- **etiquetas distintas:** se descartan todas, porque no se sabe cuál es la correcta.

La celda muestra los pares confirmados, empezando por los que tienen menos coincidencias (los más
dudosos). **Revisa los primeros:** si alguno no es la misma foto, sube `MIN_COINCIDENCIAS` (p. ej.
a 25) y vuelve a correr la celda.

In [ ]:
import cv2
from IPython.display import display
from PIL import Image

VECINAS = 10              # candidatas por foto
PISO_SIMILITUD = 0.80     # por debajo de esto ni siquiera se compara
MIN_COINCIDENCIAS = 15    # puntos que deben encajar para decir "es la misma foto"
LADO_DUP = 224

extractor = tf.keras.applications.MobileNetV2(
    input_shape=(LADO_DUP, LADO_DUP, 3), include_top=False, weights='imagenet', pooling='avg',
)

def _leer(ruta):
    img = tf.io.decode_image(tf.io.read_file(ruta), channels=3, expand_animations=False)
    img = tf.image.resize(tf.cast(img, tf.float32), (LADO_DUP, LADO_DUP))
    return tf.keras.applications.mobilenet_v2.preprocess_input(img)

lista = [(etiqueta, ruta) for etiqueta, rutas in sorted(por_clase.items()) for ruta in sorted(rutas, key=clave)]
ds_dup = (tf.data.Dataset.from_tensor_slices([r for _, r in lista])
          .map(_leer, num_parallel_calls=tf.data.AUTOTUNE).batch(64))
vectores = extractor.predict(ds_dup, verbose=0)
vectores /= np.linalg.norm(vectores, axis=1, keepdims=True) + 1e-8
similitud = vectores @ vectores.T
del extractor

# Paso 1: candidatas
candidatos = set()
for i in range(len(lista)):
    for j in np.argsort(-similitud[i])[1:VECINAS + 1]:
        if similitud[i, j] >= PISO_SIMILITUD:
            candidatos.add((min(i, int(j)), max(i, int(j))))

# Paso 2: verificación geométrica con ORB + RANSAC
orb = cv2.ORB_create(nfeatures=1000)
emparejador = cv2.BFMatcher(cv2.NORM_HAMMING)
_rasgos = {}

def rasgos(k):
    if k not in _rasgos:
        gris = cv2.imread(lista[k][1], cv2.IMREAD_GRAYSCALE)
        escala = 320 / max(gris.shape)
        gris = cv2.resize(gris, None, fx=escala, fy=escala, interpolation=cv2.INTER_AREA)
        _rasgos[k] = orb.detectAndCompute(gris, None)
    return _rasgos[k]

def coincidencias(i, j):
    (kp1, d1), (kp2, d2) = rasgos(i), rasgos(j)
    if d1 is None or d2 is None or len(kp1) < 8 or len(kp2) < 8:
        return 0
    pares_knn = [p for p in emparejador.knnMatch(d1, d2, k=2) if len(p) == 2]
    buenos = [m for m, n in pares_knn if m.distance < 0.75 * n.distance]   # prueba de Lowe
    if len(buenos) < 8:
        return 0
    origen = np.float32([kp1[m.queryIdx].pt for m in buenos])
    destino = np.float32([kp2[m.trainIdx].pt for m in buenos])
    # Solo escala, rotación y desplazamiento: lo que le pasa a una foto al recortarla o reescalarla
    _, encajan = cv2.estimateAffinePartial2D(origen, destino, method=cv2.RANSAC, ransacReprojThreshold=4.0)
    return int(encajan.sum()) if encajan is not None else 0

confirmados = []
for i, j in sorted(candidatos):
    n = coincidencias(i, j)
    if n >= MIN_COINCIDENCIAS:
        confirmados.append((i, j, n))
_rasgos.clear()

# Agrupar: si A~B y B~C, las tres son la misma foto
grupo = list(range(len(lista)))
def raiz(k):
    while grupo[k] != k:
        grupo[k] = grupo[grupo[k]]
        k = grupo[k]
    return k
for i, j, _ in confirmados:
    grupo[raiz(i)] = raiz(j)

miembros = {}
for k in range(len(lista)):
    miembros.setdefault(raiz(k), []).append(k)

descartar = set()
grupos_conflicto = grupos_misma = 0
for ks in miembros.values():
    if len(ks) < 2:
        continue
    if len({lista[k][0] for k in ks}) > 1:
        descartar.update(ks)
        grupos_conflicto += 1
    else:
        descartar.update(ks[1:])
        grupos_misma += 1

print(f"Pares candidatos revisados: {len(candidatos)}")
print(f"Confirmados como la misma foto (>= {MIN_COINCIDENCIAS} puntos): {len(confirmados)}")
print(f"Grupos con la misma etiqueta (se deja una): {grupos_misma}")
print(f"Grupos con etiquetas distintas (se descartan todas): {grupos_conflicto}")
print(f"Imágenes descartadas: {len(descartar)}")

# Los confirmados, de menos a más coincidencias: los primeros son los dudosos
confirmados.sort(key=lambda t: t[2])
pares = [(i, j) for i, j, _ in confirmados]
puntos = {(i, j): n for i, j, n in confirmados}
MAX_PARES = 12
if pares:
    lado = 150
    lienzo = Image.new('RGB', (2 * lado + 10, (lado + 18) * min(len(pares), MAX_PARES)), 'white')
    from PIL import ImageDraw
    dibujo = ImageDraw.Draw(lienzo)
    for f, (i, j) in enumerate(pares[:MAX_PARES]):
        y = f * (lado + 18)
        for c, k in enumerate((i, j)):
            img = Image.open(lista[k][1]).convert('RGB').resize((lado, lado))
            lienzo.paste(img, (c * (lado + 10), y + 16))
        texto = f"{puntos[(i, j)]} puntos  {lista[i][0].split('___')[1]} / {lista[j][0].split('___')[1]}"
        dibujo.text((2, y + 2), texto, fill='black')
    display(lienzo)

for k in descartar:
    etiqueta, ruta = lista[k]
    por_clase[etiqueta].remove(ruta)

print()
for etiqueta, rutas in sorted(por_clase.items()):
    print(f"  {len(rutas):5d}  {etiqueta}")

### 3.2 Revisión visual: quitar dibujos, collages e infografías

El dataset trae imágenes que no son fotos de una vaca: dibujos animados, figuras de artículos
científicos con paneles A/B/C/D, infografías con texto. El Grad-CAM mostró que algunas pesan en
la decisión. No hay forma confiable de detectarlas solas, así que se revisan a mano: son ~1 000
miniaturas y lleva unos 10 minutos.

1. Corre la celda de mosaicos. Cada miniatura tiene un ID: `L` para LSD y `S` para sanas.
2. Anota los IDs de lo que **no** sea una foto real de ganado.
3. Pégalos en `EXCLUIR` en la celda siguiente y córrela.

La lista se guarda en Drive (`ganado/exclusiones.json`) con la ruta de cada archivo, no con el ID.
En la próxima sesión se aplica sola y no hay que volver a revisar.

> Corre estas celdas **en orden** desde la sección 3. Los IDs dependen de las fotos que quedaron
> después de los pasos anteriores.

In [ ]:
from PIL import ImageDraw

LADO_MINI = 120
POR_FILA = 10
POR_MOSAICO = 100

ids = {}
for etiqueta in sorted(por_clase):
    prefijo = 'L' if etiqueta == 'Cattle___Lumpy_skin_disease' else 'S'
    for n, ruta in enumerate(sorted(por_clase[etiqueta], key=clave)):
        ids[f"{prefijo}{n:04d}"] = (etiqueta, ruta)

def mosaico(elementos):
    filas = -(-len(elementos) // POR_FILA)
    lienzo = Image.new('RGB', (POR_FILA * LADO_MINI, filas * (LADO_MINI + 14)), 'white')
    dibujo = ImageDraw.Draw(lienzo)
    for k, (id_, (_, ruta)) in enumerate(elementos):
        img = Image.open(ruta).convert('RGB')
        img.thumbnail((LADO_MINI - 4, LADO_MINI - 4))
        x, y = (k % POR_FILA) * LADO_MINI, (k // POR_FILA) * (LADO_MINI + 14)
        lienzo.paste(img, (x + 2, y + 14))
        dibujo.text((x + 2, y + 1), id_, fill='black')
    return lienzo

for etiqueta in sorted(por_clase):
    elementos = [(i, v) for i, v in ids.items() if v[0] == etiqueta]
    for inicio in range(0, len(elementos), POR_MOSAICO):
        parte = elementos[inicio:inicio + POR_MOSAICO]
        print(f"{etiqueta}: {parte[0][0]} a {parte[-1][0]}")
        display(mosaico(parte))

In [ ]:
EXCLUIR = []   # IDs de los mosaicos de arriba, p. ej. ['L0012', 'S0450']

ARCHIVO_EXCLUSIONES = os.path.join(GANADO_DIR, 'exclusiones.json')
previas = set()
if os.path.isfile(ARCHIVO_EXCLUSIONES):
    with open(ARCHIVO_EXCLUSIONES) as f:
        previas = set(json.load(f))

desconocidos = [i for i in EXCLUIR if i not in ids]
assert not desconocidos, f"Estos IDs no existen en los mosaicos: {desconocidos}"

excluir = previas | {clave(ids[i][1]) for i in EXCLUIR}
with open(ARCHIVO_EXCLUSIONES, 'w') as f:
    json.dump(sorted(excluir), f, indent=1, ensure_ascii=False)

antes = sum(len(v) for v in por_clase.values())
por_clase = {e: [r for r in rutas if clave(r) not in excluir] for e, rutas in por_clase.items()}
despues = sum(len(v) for v in por_clase.values())

print(f"Exclusiones guardadas: {len(excluir)} ({len(previas)} de sesiones anteriores)")
print(f"Imágenes quitadas ahora: {antes - despues}\n")
for etiqueta, rutas in sorted(por_clase.items()):
    print(f"  {len(rutas):5d}  {etiqueta}")

In [ ]:
PROPORCION_VAL = 0.15
PROPORCION_TEST = 0.15

if os.path.isdir(SPLIT_DIR):
    shutil.rmtree(SPLIT_DIR)

rng = random.Random(SEED)
conteo = {}
manifiesto = {}

for etiqueta, rutas in sorted(por_clase.items()):
    rutas = sorted(rutas)          # ordenar antes de barajar: el split no depende del orden de os.walk
    rng.shuffle(rutas)
    n_test = round(len(rutas) * PROPORCION_TEST)
    n_val = round(len(rutas) * PROPORCION_VAL)
    partes = {
        'test': rutas[:n_test],
        'val': rutas[n_test:n_test + n_val],
        'train': rutas[n_test + n_val:],
    }
    for parte, lista in partes.items():
        destino = os.path.join(SPLIT_DIR, parte, etiqueta)
        os.makedirs(destino, exist_ok=True)
        for i, src in enumerate(lista):
            shutil.copy2(src, os.path.join(destino, f"{i:04d}{os.path.splitext(src)[1].lower()}"))
        conteo[(parte, etiqueta)] = len(lista)
        manifiesto.setdefault(parte, {})[etiqueta] = [clave(r) for r in lista]

# Si ya había un split guardado (de una sesión anterior), avisar si cambió. Importa cuando se
# vuelve a correr esta sección para usar un modelo ya entrenado (6.1, 6.2): tiene que ser el
# mismo split con el que se entrenó.
RUTA_MANIFIESTO = os.path.join(GANADO_DIR, 'split_manifiesto.json')
if os.path.isfile(RUTA_MANIFIESTO):
    with open(RUTA_MANIFIESTO, encoding='utf-8') as f:
        anterior = json.load(f)
    if anterior == manifiesto:
        print("El split es idéntico al guardado en Drive.\n")
    else:
        print("⚠️  El split CAMBIÓ respecto al guardado en Drive (otras exclusiones o fotos extra).")
        print("    Un modelo entrenado antes no se puede evaluar con este split: vuelve a entrenar.\n")

with open(RUTA_MANIFIESTO, 'w', encoding='utf-8') as f:
    json.dump(manifiesto, f, indent=1, ensure_ascii=False)

etiquetas_ordenadas = sorted(por_clase)
print(f"{'':12s}" + "".join(f"{e:>30s}" for e in etiquetas_ordenadas))
for parte in ['train', 'val', 'test']:
    print(f"{parte:12s}" + "".join(f"{conteo[(parte, e)]:30d}" for e in etiquetas_ordenadas))
print("\nManifiesto guardado en", os.path.join(GANADO_DIR, 'split_manifiesto.json'))

### Mira las fotos antes de entrenar

Un modelo de 2 clases puede llegar a 99 % aprendiendo algo que no es la enfermedad: el fondo, la
iluminación o la raza, si todas las fotos de una clase vienen de un mismo lugar. Revisa que lo que
separa las dos filas de abajo sea la piel, y no otra cosa.

In [ ]:
import matplotlib.pyplot as plt

POR_CLASE = 6
fig, ejes = plt.subplots(len(etiquetas_ordenadas), POR_CLASE, figsize=(2.2 * POR_CLASE, 2.4 * len(etiquetas_ordenadas)))
for fila, etiqueta in enumerate(etiquetas_ordenadas):
    carpeta = os.path.join(SPLIT_DIR, 'train', etiqueta)
    muestra = random.Random(SEED).sample(sorted(os.listdir(carpeta)), POR_CLASE)
    for col, nombre in enumerate(muestra):
        eje = ejes[fila][col]
        eje.imshow(plt.imread(os.path.join(carpeta, nombre)))
        eje.axis('off')
        if col == 0:
            eje.set_title(etiqueta, loc='left', fontsize=9)
plt.tight_layout()
plt.show()

## 4. Pipeline de datos

Los conjuntos se cargan sin agrupar en lotes, se guardan en caché en RAM (~1 000 imágenes caben de
sobra; las de cultivos no cabían) y **después** se barajan y agrupan. Así cada época ve lotes
distintos y el augmentation cambia en cada pasada.

In [ ]:
IMG_SIZE = (224, 224)   # entrada estándar de MobileNetV2; la app la lee del propio modelo
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def cargar(parte):
    return tf.keras.utils.image_dataset_from_directory(
        os.path.join(SPLIT_DIR, parte),
        image_size=IMG_SIZE,
        batch_size=None,      # sin lotes: se agrupan después del caché
        shuffle=False,
    )

train_raw, val_raw, test_raw = cargar('train'), cargar('val'), cargar('test')

class_names = train_raw.class_names
assert val_raw.class_names == class_names and test_raw.class_names == class_names
num_classes = len(class_names)
print("Clases, en el orden que tendrá labels_ganado.txt:", class_names)

# Pesos por clase: cada clase aporta lo mismo a la pérdida aunque tenga menos fotos.
# Sin esto, decir siempre "sano" ya acierta ~68 % y el modelo puede quedarse ahí.
n_train = np.array([conteo[('train', c)] for c in class_names])
class_weight = {i: float(n_train.sum() / (num_classes * n)) for i, n in enumerate(n_train)}
print("Pesos por clase:", {class_names[i]: round(w, 2) for i, w in class_weight.items()})

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomContrast(0.2),
    layers.RandomBrightness(0.2, value_range=(0, 255)),
], name='data_augmentation')

# (pixel - 127.5) / 127.5 -> [-1, 1]. La app hace exactamente esta cuenta en
# ImageClassifier.prepararEntrada, con MEDIA = DESVIACION = 127.5. Si cambias de backbone,
# hay que cambiar también esas constantes en Kotlin.
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

def prepare(ds, training=False):
    ds = ds.cache()
    if training:
        ds = ds.shuffle(2000, seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH_SIZE)
    if training:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda x, y: (preprocess_input(x), y), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

train_ds = prepare(train_raw, training=True)
val_ds = prepare(val_raw)
test_ds = prepare(test_raw)

## 5. Modelo — MobileNetV2 (freeze → fine-tune)

Igual que cultivos. Dos cambios por el tamaño del dataset: más *dropout* (0.3) y el *early
stopping* vigila la **pérdida** de validación, no el accuracy. Con clases desbalanceadas y un
conjunto de validación de ~150 fotos, el accuracy salta mucho de una época a otra.

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights='imagenet',
)
base_model.trainable = False  # Fase 1: congelada

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()

In [ ]:
EPOCHS_HEAD = 15

history_head = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_HEAD,
    class_weight=class_weight,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
        tf.keras.callbacks.ModelCheckpoint(
            os.path.join(CKPT_DIR, 'head_best.keras'), monitor='val_loss', save_best_only=True,
        ),
    ],
)

In [ ]:
# Fase 2: se descongelan solo las últimas ~30 capas de la base, con un learning rate 100 veces
# menor. Con ~700 fotos de entrenamiento, descongelar más sobreajusta enseguida.
base_model.trainable = True
FINE_TUNE_AT = len(base_model.layers) - 30
for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

EPOCHS_FINE_TUNE = 15
history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=history_head.epoch[-1] + 1 + EPOCHS_FINE_TUNE,
    initial_epoch=history_head.epoch[-1] + 1,
    class_weight=class_weight,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
        tf.keras.callbacks.ModelCheckpoint(
            os.path.join(CKPT_DIR, 'finetuned_best.keras'), monitor='val_loss', save_best_only=True,
        ),
    ],
)

## 6. Evaluación con el conjunto de prueba

El accuracy a secas engaña aquí: con el doble de sanas que de enfermas, un modelo que nunca
detecta LSD ya saca ~68 %. Por eso la celda imprime también:

- **Sensibilidad para LSD** (*recall*): de las vacas enfermas, cuántas detectó. Es el número que
  más importa para el producto, porque una vaca enferma que se da por sana no recibe atención.
- **Accuracy balanceada**: el promedio de la sensibilidad de cada clase. Una línea base que
  adivina al azar saca 50 %.
- **La matriz de confusión**, para ver en qué dirección se equivoca.

In [ ]:
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix

y_real, probabilidades = [], []
for x, y in test_ds:
    probabilidades.append(model.predict_on_batch(x))
    y_real.append(y.numpy())
y_real = np.concatenate(y_real)
probabilidades = np.concatenate(probabilidades)
y_predicho = probabilidades.argmax(axis=1)

IDX_LSD = class_names.index('Cattle___Lumpy_skin_disease')
test_acc = float((y_predicho == y_real).mean())
balanced = balanced_accuracy_score(y_real, y_predicho)
es_lsd = y_real == IDX_LSD
sensibilidad_lsd = float((y_predicho[es_lsd] == IDX_LSD).mean())
linea_base = float(np.bincount(y_real).max() / len(y_real))

print(f"Fotos de prueba: {len(y_real)}  ({int(es_lsd.sum())} con LSD)\n")
print(f"Accuracy:                       {test_acc:.4f}")
print(f"Línea base (siempre la mayoría): {linea_base:.4f}")
print(f"Accuracy balanceada:            {balanced:.4f}")
print(f"Sensibilidad para LSD:          {sensibilidad_lsd:.4f}\n")

print("Matriz de confusión (filas = real, columnas = predicho):")
matriz = confusion_matrix(y_real, y_predicho, labels=list(range(num_classes)))
ancho = max(len(c) for c in class_names) + 2
print(" " * ancho + "".join(f"{c[:28]:>30s}" for c in class_names))
for i, fila in enumerate(matriz):
    print(f"{class_names[i]:<{ancho}s}" + "".join(f"{v:30d}" for v in fila))

print("\n" + classification_report(y_real, y_predicho, target_names=class_names, digits=3))

### 6.1 ¿Dónde mira el modelo? (Grad-CAM)

En este dataset las dos clases no solo se distinguen por la piel. Las fotos con LSD son de cerca,
en granjas de la India, con razas cebú y búfalo. Las sanas son en su mayoría fotos de banco de
imágenes con razas europeas, en pasto verde y con cielo azul. Un modelo puede sacar buen accuracy
aprendiendo **el fondo** en lugar de **la enfermedad**, y fallar en cuanto vea una vaca sana en
un corral de tierra.

Grad-CAM pinta sobre cada foto las zonas que más pesaron en la decisión (rojo = más peso). Es una
cuadrícula de 7×7 agrandada, así que verás manchas, no nódulos exactos.

**Cómo leerlo:**
- ✅ **Buena señal:** el rojo cae **sobre el animal**, sobre todo en el lomo y los costados, donde
  salen los nódulos.
- ❌ **Mala señal:** el rojo cae en **el cielo, el pasto, el suelo, personas, texto, marcas de
  agua** o los bordes. Eso significa que el número de la sección 6 no vale fuera de este dataset.
- Fíjate sobre todo en **los errores**: suelen mostrar qué atajo está usando el modelo.

La figura se guarda en Drive (`ganado/gradcam.png`) para la presentación.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import colormaps

# Si el runtime se reinició: corre las secciones 1 a 4 (sin entrenar) y esta celda carga el
# mejor modelo guardado durante el fine-tuning.
if 'model' not in globals():
    model = tf.keras.models.load_model(os.path.join(CKPT_DIR, 'finetuned_best.keras'))

# El modelo es: entrada -> MobileNetV2 -> GlobalAveragePooling -> Dropout -> Dense.
# Para Grad-CAM se necesita la salida de la última capa convolucional, que es la de MobileNetV2.
base = next(l for l in model.layers if isinstance(l, tf.keras.Model))
gap = next(l for l in model.layers if isinstance(l, layers.GlobalAveragePooling2D))
densa = next(l for l in model.layers if isinstance(l, layers.Dense))

def grad_cam(imagen, clase):
    # imagen: (224, 224, 3) en 0-255. Devuelve un mapa (224, 224) en [0, 1].
    x = preprocess_input(tf.convert_to_tensor(imagen[None, ...], tf.float32))
    with tf.GradientTape() as tape:
        mapas = base(x, training=False)          # (1, 7, 7, 1280)
        tape.watch(mapas)
        puntaje = densa(gap(mapas))[:, clase]     # el dropout no actúa al predecir
    gradientes = tape.gradient(puntaje, mapas)
    pesos = tf.reduce_mean(gradientes, axis=(1, 2))          # importancia de cada canal
    cam = tf.nn.relu(tf.reduce_sum(mapas * pesos[:, None, None, :], axis=-1))[0]
    cam = cam / (tf.reduce_max(cam) + 1e-8)
    return tf.image.resize(cam[..., None], IMG_SIZE).numpy()[..., 0]

# Imágenes de prueba sin preprocesar (0-255), en el mismo orden que en la sección 6
imagenes = np.stack([img.numpy() for img, _ in test_raw])
reales = np.array([int(y) for _, y in test_raw])
probs = model.predict(preprocess_input(imagenes.copy()), verbose=0)
predichas = probs.argmax(axis=1)

IDX_LSD = class_names.index('Cattle___Lumpy_skin_disease')
grupos = [
    ('LSD detectada',          (reales == IDX_LSD) & (predichas == IDX_LSD)),
    ('LSD que se escapó',      (reales == IDX_LSD) & (predichas != IDX_LSD)),
    ('Sana marcada como LSD',  (reales != IDX_LSD) & (predichas == IDX_LSD)),
    ('Sana bien clasificada',  (reales != IDX_LSD) & (predichas != IDX_LSD)),
]
POR_GRUPO = 5
filas = [(nombre, np.flatnonzero(m)[:POR_GRUPO]) for nombre, m in grupos if m.any()]

calor = colormaps['jet']
fig = plt.figure(figsize=(2.6 * POR_GRUPO, 3.1 * len(filas)), layout='constrained')
for subfig, (nombre, indices) in zip(np.atleast_1d(fig.subfigures(len(filas), 1)), filas):
    subfig.suptitle(nombre, x=0.01, ha='left', fontsize=11, fontweight='bold')
    for c, eje in enumerate(subfig.subplots(1, POR_GRUPO)):
        eje.axis('off')
        if c >= len(indices):
            continue
        i = indices[c]
        mapa = grad_cam(imagenes[i], int(predichas[i]))   # lo que llevó a SU decisión
        eje.imshow(imagenes[i].astype(np.uint8))
        eje.imshow(calor(mapa)[..., :3], alpha=0.45)
        eje.set_title(f"{probs[i].max():.0%} {class_names[predichas[i]].split('___')[1]}", fontsize=8)
plt.savefig(os.path.join(GANADO_DIR, 'gradcam.png'), dpi=150, bbox_inches='tight')
plt.show()

for nombre, m in grupos:
    print(f"{int(m.sum()):4d}  {nombre}")
print("\nFigura guardada en", os.path.join(GANADO_DIR, 'gradcam.png'))

### 6.2 Umbral de decisión para LSD

Por defecto, el modelo dice "LSD" cuando le da a esa clase más de 50%. Para un tamizaje ese corte
no tiene por qué ser el mejor: **que se escape una vaca enferma es peor que revisar una sana**.
En la sección 6.1 varios errores estaban cerca del 50%.

Esta celda busca los umbrales que alcanzan la sensibilidad objetivo (`OBJETIVO_SENSIBILIDAD`) con
las menos falsas alarmas. Suelen formar un tramo, y se toma **el del centro** y no el borde: con
tan pocas fotos, el borde exacto depende de una o dos imágenes.

- Se elige con el conjunto de **validación**. Si se eligiera con el de prueba, el número de prueba
  quedaría inflado, porque el umbral se habría ajustado a esas mismas fotos.
- Después se reporta qué pasa en **prueba** con el umbral de siempre (50%) y con el elegido.
- Se guarda en Drive (`export/umbral_ganado.json`) para usarlo en la app.

Validación tiene pocas fotos con LSD, así que el umbral es aproximado: tómalo como punto de partida,
no como valor exacto.

> Si el runtime se reinició: corre las secciones 1 a 4 (sin entrenar) y esta celda carga el modelo
> guardado. La sección 3 te avisa si el split no es el mismo con el que se entrenó.

In [ ]:
import matplotlib.pyplot as plt

OBJETIVO_SENSIBILIDAD = 0.90

if 'model' not in globals():
    model = tf.keras.models.load_model(os.path.join(CKPT_DIR, 'finetuned_best.keras'))

IDX_LSD = class_names.index('Cattle___Lumpy_skin_disease')

def prob_lsd(ds):
    es_lsd, prob = [], []
    for x, y in ds:
        prob.append(model.predict_on_batch(x)[:, IDX_LSD])
        es_lsd.append(y.numpy() == IDX_LSD)
    return np.concatenate(es_lsd), np.concatenate(prob)

def medir(es_lsd, prob, umbral):
    pred = prob >= umbral
    detectadas, marcadas = int((pred & es_lsd).sum()), int((pred & ~es_lsd).sum())
    n_lsd, n_sanas = int(es_lsd.sum()), int((~es_lsd).sum())
    sens, fa = detectadas / max(n_lsd, 1), marcadas / max(n_sanas, 1)
    return {
        'umbral': round(float(umbral), 2),
        'sensibilidad': round(sens, 4),
        'falsas_alarmas': round(fa, 4),
        'accuracy': round(float((pred == es_lsd).mean()), 4),
        'accuracy_balanceada': round((sens + 1 - fa) / 2, 4),
        'lsd_detectadas': f"{detectadas}/{n_lsd}",
        'sanas_marcadas_lsd': f"{marcadas}/{n_sanas}",
    }

val_lsd, val_prob = prob_lsd(val_ds)
test_lsd, test_prob = prob_lsd(test_ds)

umbrales = np.round(np.arange(0.05, 0.951, 0.01), 2)
curva = [medir(val_lsd, val_prob, u) for u in umbrales]

# De los umbrales que cumplen el objetivo, los de mejor accuracy balanceada (menos falsas
# alarmas). Suelen ser un tramo plano; se toma el del CENTRO del tramo y no el borde, porque con
# tan pocas fotos el borde exacto es casualidad y en fotos nuevas puede caer del otro lado.
cumplen = [m for m in curva if m['sensibilidad'] >= OBJETIVO_SENSIBILIDAD]
if cumplen:
    criterio = f"centro de los umbrales con sensibilidad >= {OBJETIVO_SENSIBILIDAD:.0%} y menos falsas alarmas (validación)"
else:
    cumplen = curva
    criterio = "ningún umbral llega al objetivo en validación; centro de los de mejor accuracy balanceada"
mejor = max(m['accuracy_balanceada'] for m in cumplen)
empatados = [m for m in cumplen if m['accuracy_balanceada'] == mejor]
elegido = empatados[len(empatados) // 2]
UMBRAL_LSD = elegido['umbral']

print(f"Validación: {int(val_lsd.sum())} con LSD, {int((~val_lsd).sum())} sanas")
print(f"Umbral elegido: {UMBRAL_LSD:.2f}  ({criterio})\n")

columnas = ['umbral', 'sensibilidad', 'falsas_alarmas', 'accuracy_balanceada', 'lsd_detectadas', 'sanas_marcadas_lsd']
def tabla(titulo, filas):
    print(titulo)
    print("  " + "".join(f"{c:>22s}" for c in columnas))
    for f in filas:
        print("  " + "".join(f"{f[c]:>22.2%}" if isinstance(f[c], float) and c != 'umbral'
                             else f"{str(f[c]):>22s}" for c in columnas))
    print()

referencia = sorted({0.3, 0.4, 0.5, UMBRAL_LSD})
tabla("VALIDACIÓN (con esto se elige):", [medir(val_lsd, val_prob, u) for u in referencia])

prueba_50 = medir(test_lsd, test_prob, 0.5)
prueba_elegido = medir(test_lsd, test_prob, UMBRAL_LSD)
tabla("PRUEBA (fotos que no se usaron para elegir):", [prueba_50, prueba_elegido] if UMBRAL_LSD != 0.5 else [prueba_50])

# ¿Vale la pena cambiar el umbral de la app? Solo si en prueba se nota.
mas_lsd = int(((test_prob >= UMBRAL_LSD) & test_lsd).sum() - ((test_prob >= 0.5) & test_lsd).sum())
mas_alarmas = int(((test_prob >= UMBRAL_LSD) & ~test_lsd).sum() - ((test_prob >= 0.5) & ~test_lsd).sum())
print(f"En prueba, {UMBRAL_LSD:.2f} frente a 0.50: {mas_lsd:+d} vacas con LSD detectadas, {mas_alarmas:+d} falsas alarmas.")
if mas_lsd <= 0:
    print("-> No mejora la detección en prueba: conviene dejar 0.50 en la app (no requiere cambios).\n")
else:
    print("-> Mejora la detección en prueba: vale la pena usar este umbral en la app.\n")

# Curva en validación, para la presentación
fig, eje = plt.subplots(figsize=(7, 3.6))
eje.plot(umbrales, [m['sensibilidad'] for m in curva], label='Sensibilidad para LSD')
eje.plot(umbrales, [m['falsas_alarmas'] for m in curva], label='Falsas alarmas (sanas marcadas)')
eje.axvline(0.5, color='gray', linestyle=':', label='Umbral por defecto (0.50)')
eje.axvline(UMBRAL_LSD, color='crimson', linestyle='--', label=f'Umbral elegido ({UMBRAL_LSD:.2f})')
eje.set_xlabel('Umbral para decir "LSD"')
eje.set_ylim(-0.02, 1.02)
eje.set_title('Validación: qué se gana y qué se pierde al mover el umbral')
eje.legend(loc='center left', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(GANADO_DIR, 'umbral.png'), dpi=150)
plt.show()

with open(os.path.join(EXPORT_DIR, 'umbral_ganado.json'), 'w', encoding='utf-8') as f:
    json.dump({
        'clase': 'Cattle___Lumpy_skin_disease',
        'umbral': UMBRAL_LSD,
        'criterio': criterio,
        'objetivo_sensibilidad': OBJETIVO_SENSIBILIDAD,
        'validacion': elegido,
        'prueba': {'umbral_0.50': prueba_50, 'umbral_elegido': prueba_elegido},
    }, f, indent=1, ensure_ascii=False)
print("Guardado en", os.path.join(EXPORT_DIR, 'umbral_ganado.json'))

### Qué decir de este número en la presentación

- Es una **prueba de concepto**: ~1 000 fotos de **una sola fuente**, con una prueba de ~150.
  Un punto porcentual equivale a 1 o 2 fotos.
- A diferencia de cultivos, **no hay un conjunto de campo** equivalente a PlantDoc. El número es de
  laboratorio y en campo va a bajar, igual que en cultivos.
- Reporta la **sensibilidad para LSD** junto al accuracy, y di sobre cuántas fotos se midió.
- Licencia CC BY 4.0: la cita del dataset (arriba) tiene que aparecer en la documentación técnica.

## 7. Exportar a TFLite y verificar el contrato con la app

Cuantización **float16**: los pesos se guardan a media precisión (~4.6 MB) y la entrada y la
salida siguen en float32. No es la misma que cultivos (int8 dinámica, ~2.4 MB) por una razón
medida: con int8, el `.tflite` cambió la decisión en 3 de 128 fotos de prueba. Este modelo deja
varios casos cerca del 50% y ahí cualquier corrimiento cambia la decisión. Con float16, las
probabilidades quedan prácticamente iguales a las de Keras.

La app no necesita metadata, pero sí que se cumplan cuatro cosas, y la segunda celda las comprueba:

1. Entrada `[1, alto, ancho, 3]` en `float32`. La app toma el lado del propio modelo.
2. Salida `[1, clases]`.
3. `labels_ganado.txt` con **una etiqueta por salida**, en el mismo orden. Si no coincide, la app
   desactiva el clasificador en vez de etiquetar mal.
4. Que el `.tflite` cuantizado prediga lo mismo que el modelo de Keras.

In [ ]:
# Si el runtime se reinició: corre las secciones 1 a 4 (sin entrenar) y esta celda exporta el
# modelo guardado durante el fine-tuning.
if 'model' not in globals():
    model = tf.keras.models.load_model(os.path.join(CKPT_DIR, 'finetuned_best.keras'))

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]   # pesos en float16; entrada/salida float32
tflite_model = converter.convert()

TFLITE_PATH = os.path.join(EXPORT_DIR, 'agrosense_ganado_model.tflite')
LABELS_PATH = os.path.join(EXPORT_DIR, 'labels_ganado.txt')

with open(TFLITE_PATH, 'wb') as f:
    f.write(tflite_model)
with open(LABELS_PATH, 'w') as f:
    f.write('\n'.join(class_names) + '\n')

print(f"Modelo:    {TFLITE_PATH} ({os.path.getsize(TFLITE_PATH) / 2**20:.2f} MB)")
print(f"Etiquetas: {LABELS_PATH}")
for i, c in enumerate(class_names):
    print(f"  {i}: {c}")

In [ ]:
# tf.lite.Interpreter está marcado como obsoleto y TensorFlow lo va a quitar; su reemplazo es
# LiteRT. Se usa el que exista, para que esta celda no se rompa cuando Colab actualice TF.
Interpreter = getattr(tf.lite, 'Interpreter', None)
if Interpreter is None:
    !pip install -q ai-edge-litert
    from ai_edge_litert.interpreter import Interpreter

interprete = Interpreter(model_path=TFLITE_PATH)
interprete.allocate_tensors()
entrada = interprete.get_input_details()[0]
salida = interprete.get_output_details()[0]

# 1 y 2: forma y tipo, como los lee ImageClassifier.kt
assert entrada['dtype'] == np.float32, f"La entrada es {entrada['dtype']}; la app manda float32."
assert list(entrada['shape']) == [1, IMG_SIZE[0], IMG_SIZE[1], 3], f"Entrada inesperada: {entrada['shape']}"
assert list(salida['shape']) == [1, num_classes], f"Salida inesperada: {salida['shape']}"

# 3: el archivo de etiquetas, leído igual que en la app (líneas no vacías)
with open(LABELS_PATH) as f:
    etiquetas = [l.strip() for l in f if l.strip()]
assert etiquetas == class_names, f"labels_ganado.txt no coincide con class_names: {etiquetas}"

# 4: el .tflite cuantizado contra Keras, foto por foto del conjunto de prueba. Se compara la
# clase ganadora y, si se eligió un umbral en 6.2, también la decisión con ese umbral: la
# cuantización mueve un poco las probabilidades y puede cambiar casos justo en el corte.
umbral = globals().get('UMBRAL_LSD')
idx_lsd = class_names.index('Cattle___Lumpy_skin_disease')
coinciden = coinciden_umbral = total = 0
for x, _ in test_ds:
    keras_prob = model.predict_on_batch(x)
    for i in range(x.shape[0]):
        interprete.set_tensor(entrada['index'], x[i:i + 1].numpy().astype(np.float32))
        interprete.invoke()
        tflite_prob = interprete.get_tensor(salida['index'])[0]
        coinciden += int(tflite_prob.argmax() == keras_prob[i].argmax())
        if umbral is not None:
            coinciden_umbral += int((tflite_prob[idx_lsd] >= umbral) == (keras_prob[i][idx_lsd] >= umbral))
        total += 1

acuerdo = coinciden / total
print(f"El .tflite coincide con Keras en {coinciden}/{total} fotos ({acuerdo:.1%}).")
assert acuerdo >= 0.98, "La cuantización cambió demasiadas predicciones; NO copies este modelo a la app."
if umbral is not None:
    acuerdo_umbral = coinciden_umbral / total
    print(f"Con el umbral {umbral:.2f}: coincide en {coinciden_umbral}/{total} fotos ({acuerdo_umbral:.1%}).")
    assert acuerdo_umbral >= 0.98, "Con el umbral elegido la cuantización cambia demasiadas decisiones."
print("\nOK — el modelo cumple el contrato de ImageClassifier.kt.")

## 8. Siguiente paso en Android Studio

0. **Umbral:** `export/umbral_ganado.json` (sección 6.2) trae el umbral para decir "LSD". La app
   hoy decide por la clase con mayor probabilidad, que equivale a un umbral de 0.50. Para usar el
   elegido, `ImageClassifier` tiene que recibirlo como parámetro y marcar LSD cuando
   `P(LSD) >= umbral`. Va junto con los cambios del punto 2.
1. Descarga de Drive `export/agrosense_ganado_model.tflite` y `export/labels_ganado.txt`, y
   cópialos **juntos** a `app/src/main/assets/`, al lado de los de cultivos.

   **No los renombres** a `agrosense_crop_model.tflite` / `labels.txt`: reemplazarías el modelo de
   cultivos.
2. La app todavía no los va a usar sola. `ImageClassifier` tiene los nombres de archivo fijos
   (`MODELO` y `ETIQUETAS`), así que hace falta:
   - que reciba el modelo y las etiquetas como parámetro, para crear uno para cultivos y otro para
     ganado;
   - el campo de módulo en `Diagnostico` (tarea 30), que necesita la migración de Room 2 → 3;
   - la pantalla para elegir cultivo o ganado (tarea 31).
3. Para la documentación (tarea 46): la cita del dataset, los conteos del split (sección 3), los
   números de la sección 6 y el aviso de que no hay validación de campo.